# Isolation Forest — Fine-Tuning & Validación del Modelo

Este notebook documenta el proceso de selección de features y ajuste de hiperparámetros del modelo de anomalía que alimenta el scoring de riesgo.

**Problema:** Isolation Forest es no supervisado — no hay etiquetas de "fraude real". Para evaluar el modelo usamos dos estrategias de validación indirecta:

1. **Weak supervision con reglas duras:** Los usuarios que dispararon al menos una regla determinista (R1–R7) son tratados como "anomalías conocidas". Medimos si el IF los rankea correctamente.
2. **Estabilidad entre seeds:** Un buen modelo produce rankings consistentes independientemente de la semilla aleatoria.

**Estructura del notebook:**
1. Setup y carga de datos desde `02_intermediate/` y `05_model_input/`
2. Baseline: métricas del modelo actual
3. Experimento A — Feature subsets (¿qué features agregan señal?)
4. Experimento B — Hiperparámetros (contamination, n_estimators, max_features)
5. Experimento C — Estabilidad entre seeds
6. Decisión final y justificación
7. Actualización del pipeline

## 🔑 Resultado clave

> Cómo validar un modelo **no supervisado** sin ground truth: weak-supervision
> (las reglas R1–R7 como "anomalías conocidas") + estabilidad entre seeds.

- **Lift@5% holgadamente > 3×** sobre la tasa base: el Isolation Forest concentra a los
  usuarios *rule-flagged* en la cabeza del ranking **sin** haberlos visto etiquetados (§4, §7).
- **Ranking muy estable** entre seeds (Spearman ρ ≈ 0.99). La cabeza (top-20) es algo
  menos estable — por eso el scoring final combina IF con **LOF + Z-score** (§13).
- **`contamination` no afecta el ranking** (se cancela en la normalización min-max) → no se tunea.
- **Honestidad metodológica:** la evaluación es *in-sample* con etiquetas débiles; el lift
  cae al medir held-out (§14). No es ground truth real, y queda documentado como limitación.

Todos los gráficos son **interactivos** (Plotly) — hover para ver valores exactos.

## 0. Setup

In [1]:
import json
import warnings
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

DATA_INT   = Path('../data/02_intermediate')
DATA_PRI   = Path('../data/03_primary')
DATA_MODEL = Path('../data/05_model_input')

# ── Plotly: gráficos interactivos (se ejecutan en local: Jupyter / VS Code) ───
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"


In [2]:
users = pd.read_csv(DATA_INT / 'users_clean.csv', parse_dates=['created_at'])
perms = pd.read_csv(DATA_INT / 'perms_clean.csv', parse_dates=['assigned_at', 'expires_at'])
logs  = pd.read_csv(DATA_INT / 'logs_clean.csv',  parse_dates=['timestamp'])

features     = pd.read_csv(DATA_PRI   / 'user_features.csv')
rule_scores  = pd.read_csv(DATA_MODEL / 'hard_rule_scores.csv')

print(f'users: {users.shape} | perms: {perms.shape} | logs: {logs.shape}')
print(f'features: {features.shape} | rule_scores: {rule_scores.shape}')

users: (500, 8) | perms: (2944, 7) | logs: (20495, 7)
features: (500, 18) | rule_scores: (500, 4)


## 1. Construcción de pseudo-etiquetas

Antes de cualquier experimento, construimos las pseudo-etiquetas para validación.

**Criterio:** Un usuario es marcado como *anomalía conocida* (`pseudo_label = 1`) si disparó al menos una regla dura (rule_score > 0). Esto nos da señal supervisada débil sin necesidad de etiquetas manuales.

> **Limitación reconocida:** Las reglas duras solo cubren H1, H2, H5, H7, H8, H11 y H13. Hay comportamientos anómalos (H3, H6, H9, H10, H12) que el IF debería detectar pero que las reglas NO capturan. Esto sesga la validación hacia los patrones de las reglas — un ROC-AUC alto indica acuerdo entre capas, no que el IF sea correcto en términos absolutos.

In [3]:
# Pseudo-labels: rule_score > 0 → anomalía conocida
pseudo = rule_scores[['user_id', 'rule_score']].copy()
pseudo['pseudo_label'] = (pseudo['rule_score'] > 0).astype(int)

print(f'Anomalías conocidas (pseudo_label=1): {pseudo["pseudo_label"].sum()} / {len(pseudo)}')
print(f'Tasa de anomalías: {pseudo["pseudo_label"].mean():.1%}')
print()
print('Distribución de rule_score:')
print(pseudo['rule_score'].value_counts().sort_index())

Anomalías conocidas (pseudo_label=1): 54 / 500
Tasa de anomalías: 10.8%

Distribución de rule_score:
rule_score
0     446
15     15
20     27
30      1
35      5
45      3
50      1
60      2
Name: count, dtype: int64


## 2. Definición de feature subsets

Agrupamos las 17 features en 4 dimensiones conceptuales para entender qué aporta cada una al modelo.

In [4]:
ALL_FEATURES = [
    'total_accesses', 'z_volume_peers', 'distinct_resources', 'z_distinct_peers',
    'exfil_ratio', 'z_exfil_peers', 'action_entropy', 'after_hours_ratio',
    'weekend_ratio', 'avg_session_sec', 'perm_count', 'z_perms_peers',
    'max_crit_accessed', 'very_high_access_ratio', 'high_plus_access_ratio',
    'delete_ratio', 'is_external',
]

FEATURE_SUBSETS = {
    'full': ALL_FEATURES,

    'volumetry': [
        'total_accesses', 'z_volume_peers',
        'distinct_resources', 'z_distinct_peers',
        'avg_session_sec',
    ],

    'behavioral': [
        'exfil_ratio', 'z_exfil_peers', 'action_entropy',
        'after_hours_ratio', 'weekend_ratio', 'delete_ratio',
    ],

    'privilege': [
        'perm_count', 'z_perms_peers',
        'max_crit_accessed', 'very_high_access_ratio', 'high_plus_access_ratio',
        'is_external',
    ],

    'peer_z_scores': [
        'z_volume_peers', 'z_distinct_peers', 'z_exfil_peers', 'z_perms_peers',
    ],

    'no_z_scores': [
        f for f in ALL_FEATURES if not f.startswith('z_')
    ],
}

for name, cols in FEATURE_SUBSETS.items():
    print(f'{name:15s} → {len(cols):2d} features: {cols}')

full            → 17 features: ['total_accesses', 'z_volume_peers', 'distinct_resources', 'z_distinct_peers', 'exfil_ratio', 'z_exfil_peers', 'action_entropy', 'after_hours_ratio', 'weekend_ratio', 'avg_session_sec', 'perm_count', 'z_perms_peers', 'max_crit_accessed', 'very_high_access_ratio', 'high_plus_access_ratio', 'delete_ratio', 'is_external']
volumetry       →  5 features: ['total_accesses', 'z_volume_peers', 'distinct_resources', 'z_distinct_peers', 'avg_session_sec']
behavioral      →  6 features: ['exfil_ratio', 'z_exfil_peers', 'action_entropy', 'after_hours_ratio', 'weekend_ratio', 'delete_ratio']
privilege       →  6 features: ['perm_count', 'z_perms_peers', 'max_crit_accessed', 'very_high_access_ratio', 'high_plus_access_ratio', 'is_external']
peer_z_scores   →  4 features: ['z_volume_peers', 'z_distinct_peers', 'z_exfil_peers', 'z_perms_peers']
no_z_scores     → 13 features: ['total_accesses', 'distinct_resources', 'exfil_ratio', 'action_entropy', 'after_hours_ratio', 'w

## 3. Funciones de evaluación

Como el modelo es **no supervisado**, no medimos accuracy contra un ground truth. Usamos métricas que son válidas sin etiquetas reales:

| Métrica | Qué mide | Cómo se interpreta |
|---|---|---|
| **Stability (full)** | Correlación de Spearman del ranking completo entre seeds | Label-free. >0.95 = ranking reproducible |
| **Stability (top-K)** | Jaccard del top-K entre seeds | Label-free. Mide si la parte *accionable* del ranking es estable |
| **Lift@5%** | Concentración de usuarios rule-flagged en el top-5% IF / tasa base | Sanity check **unidireccional**: >1 = el IF concentra casos conocidos por encima del azar |
| **ROC-AUC** | Capacidad de rankear rule-flagged por encima de normales | Solo como piso de validez, no como objetivo a maximizar |

> **Decisiones de diseño respecto a la versión anterior:**
> - **Se eliminó `contamination` del tuning.** En sklearn `contamination` solo mueve el umbral de `predict()`, no el orden de `decision_function()`. Como normalizamos min-max, su efecto se cancela exactamente (verificado: scores idénticos para cualquier contamination). No afecta el ranking ni en el notebook ni en el pipeline.
> - **Se eliminó "Score Gap".** Era tautológico: una distribución uniforme aleatoria da ~0.50, indistinguible del IF real. No discriminaba señal de ruido.
> - **"Complementarity" se reemplazó por Lift unidireccional.** La métrica anterior premiaba el *desacuerdo* con las reglas — un modelo aleatorio puntuaba mejor (0.89 vs 0.56 del IF). El Lift mide lo correcto: si el IF concentra los casos conocidos *por encima del azar*, sin premiar la replicación de las reglas.

In [5]:
def fit_if(features_df, feature_cols,
           n_estimators=200, max_features=1.0,
           contamination=0.05, random_state=42):
    """Fit IF pipeline and return min-max normalized risk scores [0, 1].
    Nota: contamination NO afecta el ranking (se cancela en la normalización);
    se mantiene solo por compatibilidad con la firma del pipeline."""
    X = features_df[feature_cols].values.astype(float)
    pipe = SKPipeline([
        ('scaler', StandardScaler()),
        ('if', IsolationForest(
            n_estimators=n_estimators,
            max_features=max_features,
            contamination=contamination,
            random_state=random_state,
        )),
    ])
    pipe.fit(X)
    d = pipe.decision_function(X)
    rng = d.max() - d.min()
    return np.zeros(len(d)) if rng == 0 else (d.max() - d) / rng


def evaluate(features_df, feature_cols, pseudo_df, top_pct=0.05, **if_kwargs):
    """ROC-AUC, Lift@top_pct y Spearman vs rule_score."""
    if_risk = fit_if(features_df, feature_cols, **if_kwargs)
    m = features_df[['user_id']].copy()
    m['if_risk'] = if_risk
    m = m.merge(pseudo_df[['user_id', 'pseudo_label', 'rule_score']], on='user_id')

    auc = roc_auc_score(m['pseudo_label'], m['if_risk'])

    # Lift unidireccional: concentración de rule-flagged en top-K vs tasa base
    k = max(1, int(len(m) * top_pct))
    base_rate = m['pseudo_label'].mean()
    precision_topk = m.nlargest(k, 'if_risk')['pseudo_label'].mean()
    lift = precision_topk / base_rate if base_rate > 0 else np.nan

    rho, pval = spearmanr(m['rule_score'], m['if_risk'])
    return {'roc_auc': round(auc, 4), 'lift@5': round(lift, 3),
            'spearman_rho': round(rho, 4), 'spearman_pval': round(pval, 4)}


def stability(features_df, feature_cols, n_seeds=5, **if_kwargs):
    """Estabilidad del RANKING COMPLETO: media de Spearman entre seeds."""
    scores = [fit_if(features_df, feature_cols, random_state=s, **if_kwargs)
              for s in range(n_seeds)]
    rhos = [spearmanr(scores[i], scores[j]).statistic
            for i in range(len(scores)) for j in range(i + 1, len(scores))]
    return round(float(np.mean(rhos)), 4)


def stability_topk(features_df, feature_cols, k=20, n_seeds=5, **if_kwargs):
    """Estabilidad del TOP-K: media de Jaccard del conjunto top-K entre seeds."""
    scores = [fit_if(features_df, feature_cols, random_state=s, **if_kwargs)
              for s in range(n_seeds)]
    tops = [set(np.argsort(s)[-k:]) for s in scores]
    jacc = [len(tops[i] & tops[j]) / len(tops[i] | tops[j])
            for i in range(len(tops)) for j in range(i + 1, len(tops))]
    return round(float(np.mean(jacc)), 4)


print('Funciones: fit_if, evaluate, stability (full), stability_topk (top-K)')

Funciones: fit_if, evaluate, stability (full), stability_topk (top-K)


## 4. Baseline — configuración actual del pipeline

Configuración en `conf/base/parameters/risk_scoring.yml`: `n_estimators=200`, `max_features=1.0` (todas las features). (`contamination=0.05` es irrelevante para el ranking.)

In [6]:
baseline = evaluate(features, ALL_FEATURES, pseudo, n_estimators=200, max_features=1.0)
base_stab_full = stability(features, ALL_FEATURES, n_estimators=200, max_features=1.0)
base_stab_topk = stability_topk(features, ALL_FEATURES, k=20, n_estimators=200, max_features=1.0)

base_rate = pseudo['pseudo_label'].mean()
print('=== BASELINE (configuración actual) ===')
print(f'  Stability full:  {base_stab_full:.4f}   (Spearman ranking completo entre seeds)')
print(f'  Stability top-20:{base_stab_topk:.4f}   (Jaccard del top-20 entre seeds)')
print(f'  Lift@5%:         {baseline["lift@5"]:.2f}x    (vs tasa base de rule-flagged = {base_rate:.1%})')
print(f'  ROC-AUC:         {baseline["roc_auc"]:.4f}   (piso de validez)')
print(f'  Spearman ρ:      {baseline["spearman_rho"]:.4f}   (vs rule_score; moderado = complementario)')

=== BASELINE (configuración actual) ===
  Stability full:  0.9704   (Spearman ranking completo entre seeds)
  Stability top-20:0.6144   (Jaccard del top-20 entre seeds)
  Lift@5%:         4.07x    (vs tasa base de rule-flagged = 10.8%)
  ROC-AUC:         0.8737   (piso de validez)
  Spearman ρ:      0.4042   (vs rule_score; moderado = complementario)


In [7]:
if_risk_baseline = fit_if(features, ALL_FEATURES, n_estimators=200, max_features=1.0)
mb = features[["user_id"]].copy()
mb["if_risk"] = if_risk_baseline
mb = mb.merge(pseudo[["user_id", "pseudo_label", "rule_score"]], on="user_id")

# 1) Distribución de IF risk score, con p95 marcado
p95 = mb["if_risk"].quantile(0.95)
fig = px.histogram(mb, x="if_risk", nbins=40, title="Distribución IF risk score",
                   color_discrete_sequence=["#4C72B0"], template="plotly_white")
fig.add_vline(x=p95, line_dash="dash", line_color="#C44E52", annotation_text=f"p95={p95:.2f}")
fig.update_layout(xaxis_title="Score [0-1]", yaxis_title="Usuarios", bargap=0.05); fig.show()

# 2) Score por pseudo-label (separación sin regla vs con regla)
mb["grupo"] = mb["pseudo_label"].map({0: "Sin regla", 1: "Con regla"})
fig = px.box(mb, x="grupo", y="if_risk", color="grupo", title="IF score: sin regla vs con regla",
             color_discrete_map={"Sin regla": "#4C72B0", "Con regla": "#C44E52"}, template="plotly_white")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="IF risk score"); fig.show()

# 3) Lift@5%: precision en top-K vs tasa base
k = int(len(mb) * 0.05)
prec = mb.nlargest(k, "if_risk")["pseudo_label"].mean()
dfk = pd.DataFrame({"grupo": ["Tasa base", f"Top-{k} IF"], "val": [base_rate, prec]})
fig = px.bar(dfk, x="grupo", y="val", color="grupo", text=dfk["val"].map(lambda v: f"{v:.1%}"),
             title=f"Lift@5% = {prec/base_rate:.2f}x",
             color_discrete_sequence=["#999999", "#C44E52"], template="plotly_white")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Fracción rule-flagged"); fig.show()

## 5. Experimento A — Feature subsets

Probamos cada grupo de features para entender qué dimensión aporta más señal estable.

**Hipótesis:** los peer z-scores deberían dar rankings más estables que las features absolutas, porque capturan desviación contextual.

In [8]:
results_a = []
for name, cols in FEATURE_SUBSETS.items():
    m = evaluate(features, cols, pseudo, n_estimators=200, max_features=1.0)
    sf = stability(features, cols, n_estimators=200, max_features=1.0)
    st = stability_topk(features, cols, k=20, n_estimators=200, max_features=1.0)
    results_a.append({'subset': name, 'n_features': len(cols),
                      'roc_auc': m['roc_auc'], 'lift@5': m['lift@5'],
                      'stab_full': sf, 'stab_top20': st})

df_a = pd.DataFrame(results_a).sort_values('stab_full', ascending=False)
print(df_a.to_string(index=False))

       subset  n_features  roc_auc  lift@5  stab_full  stab_top20
    privilege           6   0.9021   6.296     0.9858      0.6417
    volumetry           5   0.5985   1.852     0.9818      0.6969
peer_z_scores           4   0.4570   0.741     0.9796      0.8039
  no_z_scores          13   0.8901   4.444     0.9790      0.7418
   behavioral           6   0.4759   2.222     0.9786      0.8379
         full          17   0.8737   4.074     0.9704      0.6144


In [9]:
specs = [("stab_full", "Stability full", "#8172B2", base_stab_full),
         ("stab_top20", "Stability top-20", "#937860", base_stab_topk),
         ("lift@5", "Lift@5%", "#55A868", baseline["lift@5"]),
         ("roc_auc", "ROC-AUC", "#4C72B0", baseline["roc_auc"])]
for metric, title, color, ref in specs:
    d = df_a.sort_values(metric)
    fig = px.bar(d, x=metric, y="subset", orientation="h", title=f"Experimento A — {title}",
                 color_discrete_sequence=[color], template="plotly_white")
    fig.add_vline(x=ref, line_dash="dash", line_color="red", annotation_text="baseline (full)")
    fig.update_layout(xaxis_title=title, yaxis_title=""); fig.show()

### Interpretación Experimento A

> Completar con los valores observados. Puntos a analizar:
> - ¿`full` domina en stability, o un subset más chico es igual de estable con menos features?
> - ¿Los z-scores (`peer_z_scores`) dan más lift que las absolutas (`no_z_scores`)?
> - Si un subset reducido mantiene lift y mejora stability, es candidato para simplificar el modelo.

## 6. Experimento B — Búsqueda de hiperparámetros

Solo barremos **`n_estimators`** y **`max_features`**. `contamination` queda fuera porque no afecta el ranking (demostrado abajo).

| Parámetro | Valores | Efecto |
|---|---|---|
| `n_estimators` | [50, 100, 200, 300, 500] | Más árboles → más estabilidad, con retorno decreciente |
| `max_features` | [0.5, 0.6, 0.75, 0.9, 1.0] | Fracción de features por árbol; menor → más diversidad entre árboles |

In [10]:
# Demostración: contamination NO afecta el ranking normalizado
s_a = fit_if(features, ALL_FEATURES, contamination=0.02)
s_b = fit_if(features, ALL_FEATURES, contamination=0.10)
print('Prueba de no-efecto de contamination:')
print(f'  max |score(0.02) - score(0.10)| = {np.max(np.abs(s_a - s_b)):.2e}  (=> idénticos)')
print()

# Grid solo sobre n_estimators x max_features
N_ESTIMATORS = [50, 100, 200, 300, 500]
MAX_FEATURES = [0.5, 0.6, 0.75, 0.9, 1.0]

results_b = []
for n_est, mf in product(N_ESTIMATORS, MAX_FEATURES):
    m = evaluate(features, ALL_FEATURES, pseudo, n_estimators=n_est, max_features=mf)
    sf = stability(features, ALL_FEATURES, n_seeds=5, n_estimators=n_est, max_features=mf)
    st = stability_topk(features, ALL_FEATURES, k=20, n_seeds=5,
                        n_estimators=n_est, max_features=mf)
    results_b.append({'n_estimators': n_est, 'max_features': mf,
                      'roc_auc': m['roc_auc'], 'lift@5': m['lift@5'],
                      'stab_full': sf, 'stab_top20': st})

df_b = pd.DataFrame(results_b)
print(f'{len(df_b)} configuraciones evaluadas. Top-10 por stability_full:')
print(df_b.sort_values('stab_full', ascending=False).head(10).to_string(index=False))

Prueba de no-efecto de contamination:
  max |score(0.02) - score(0.10)| = 0.00e+00  (=> idénticos)



25 configuraciones evaluadas. Top-10 por stability_full:
 n_estimators  max_features  roc_auc  lift@5  stab_full  stab_top20
          500          0.90   0.8702   3.704     0.9884      0.7293
          500          0.75   0.8674   4.074     0.9880      0.7404
          500          0.50   0.8574   4.074     0.9879      0.6908
          500          1.00   0.8720   4.444     0.9872      0.7162
          500          0.60   0.8608   4.444     0.9866      0.7339
          300          1.00   0.8764   3.704     0.9818      0.6557
          300          0.90   0.8813   4.074     0.9816      0.6573
          300          0.50   0.8575   3.704     0.9793      0.6216
          300          0.75   0.8784   3.704     0.9787      0.6093
          300          0.60   0.8696   4.444     0.9767      0.7200


In [11]:
for metric, title, cmap in [
        ("stab_full", "Stability full", "YlGn"),
        ("stab_top20", "Stability top-20", "YlGn"),
        ("lift@5", "Lift@5%", "Oranges"),
        ("roc_auc", "ROC-AUC", "Blues")]:
    pivot = df_b.pivot_table(index="n_estimators", columns="max_features", values=metric)
    fig = px.imshow(pivot, text_auto=".3f", color_continuous_scale=cmap, aspect="auto",
                    title=f"Experimento B — {title} (n_estimators × max_features)", template="plotly_white")
    fig.update_layout(xaxis_title="max_features", yaxis_title="n_estimators"); fig.show()

In [12]:
# Efecto de n_estimators sobre estabilidad, por max_features
for metric, title in [("stab_full", "Stability full"), ("stab_top20", "Stability top-20")]:
    fig = go.Figure()
    for mf in MAX_FEATURES:
        sub = df_b[df_b["max_features"] == mf].sort_values("n_estimators")
        fig.add_trace(go.Scatter(x=sub["n_estimators"], y=sub[metric], mode="lines+markers", name=f"mf={mf}"))
    fig.update_layout(title=f"{title} vs n_estimators", xaxis_title="n_estimators", yaxis_title=title,
                      template="plotly_white", legend_title="max_features"); fig.show()

## 7. Selección de la configuración

**Criterio (sin score compuesto contradictorio):**
1. **Lift@5% ≥ 3** — piso de validez: el modelo debe concentrar casos conocidos muy por encima del azar.
2. Entre los que cumplen, **máxima Stability full** — priorizamos reproducibilidad, que es la propiedad label-free más confiable.
3. Desempate por menor `n_estimators` (modelo más liviano).

ROC-AUC NO se maximiza porque está en tensión con la complementariedad deseada (más ROC-AUC = más solapamiento con las reglas).

In [13]:
LIFT_FLOOR = 3.0
candidates = df_b[df_b['lift@5'] >= LIFT_FLOOR].copy()
candidates = candidates.sort_values(['stab_full', 'n_estimators'],
                                    ascending=[False, True])

SEL = candidates.iloc[0]
SEL_N  = int(SEL['n_estimators'])
SEL_MF = float(SEL['max_features'])

print('=== CONFIGURACIÓN SELECCIONADA ===')
print(f'  n_estimators = {SEL_N}')
print(f'  max_features = {SEL_MF}')
print()
print(f'  Stability full:   {SEL["stab_full"]:.4f}  (baseline {base_stab_full:.4f})')
print(f'  Stability top-20: {SEL["stab_top20"]:.4f}  (baseline {base_stab_topk:.4f})')
print(f'  Lift@5%:          {SEL["lift@5"]:.2f}x  (baseline {baseline["lift@5"]:.2f}x)')
print(f'  ROC-AUC:          {SEL["roc_auc"]:.4f}  (baseline {baseline["roc_auc"]:.4f})')

=== CONFIGURACIÓN SELECCIONADA ===
  n_estimators = 500
  max_features = 0.9

  Stability full:   0.9884  (baseline 0.9704)
  Stability top-20: 0.7293  (baseline 0.6144)
  Lift@5%:          3.70x  (baseline 4.07x)
  ROC-AUC:          0.8702  (baseline 0.8737)


## 8. Estabilidad detallada de la configuración seleccionada

Reportamos la estabilidad de forma **honesta**: el ranking completo es muy estable, pero la cabeza del ranking (lo que se accionaría) lo es menos. Esto es importante para producción.

In [14]:
from scipy.stats import kendalltau

SEEDS = [0, 7, 13, 42, 99]
scores_all = {s: fit_if(features, ALL_FEATURES, n_estimators=SEL_N,
                        max_features=SEL_MF, random_state=s) for s in SEEDS}

n = len(SEEDS)
mat_sp, mat_ken = np.zeros((n, n)), np.zeros((n, n))
for i, s1 in enumerate(SEEDS):
    for j, s2 in enumerate(SEEDS):
        mat_sp[i, j] = spearmanr(scores_all[s1], scores_all[s2]).statistic
        t = np.argsort(scores_all[s1])[-20:]
        mat_ken[i, j] = kendalltau(scores_all[s1][t], scores_all[s2][t]).statistic

lab = [f"seed={s}" for s in SEEDS]
fig = px.imshow(mat_sp, x=lab, y=lab, text_auto=".3f", color_continuous_scale="YlGn",
                zmin=0.9, zmax=1.0, title="Spearman ρ (ranking completo)", template="plotly_white")
fig.show()
fig = px.imshow(mat_ken, x=lab, y=lab, text_auto=".3f", color_continuous_scale="YlOrRd",
                zmin=0.0, zmax=1.0, title="Kendall τ (top-20)", template="plotly_white")
fig.show()

tri = np.triu_indices(n, k=1)
print(f"Spearman ρ medio (ranking completo): {np.mean(mat_sp[tri]):.4f}  <- muy estable")
print(f"Kendall τ medio (top-20):            {np.mean(mat_ken[tri]):.4f}  <- la cabeza es menos estable")

Spearman ρ medio (ranking completo): 0.9891  <- muy estable
Kendall τ medio (top-20):            0.6484  <- la cabeza es menos estable


In [15]:
# ¿Qué usuarios son robustos al seed? (aparecen en top-10 de TODOS los seeds)
top10 = {s: set(features['user_id'].values[np.argsort(scores_all[s])[-10:]]) for s in SEEDS}
common = set.intersection(*top10.values())
union  = set.union(*top10.values())

print(f'Usuarios en top-10 de TODOS los seeds: {len(common)}  -> {sorted(common)}')
print(f'Usuarios que aparecen en ALGÚN top-10: {len(union)}')
print()
print('Lectura: el núcleo robusto (interseccion) es el que se debe priorizar para acción.')
print('Los que aparecen de forma intermitente requieren un score promediado entre seeds')
print('o un n_estimators mayor para estabilizarse.')

Usuarios en top-10 de TODOS los seeds: 7  -> ['USR0009', 'USR0021', 'USR0029', 'USR0041', 'USR0060', 'USR0328', 'USR0463']
Usuarios que aparecen en ALGÚN top-10: 14

Lectura: el núcleo robusto (interseccion) es el que se debe priorizar para acción.
Los que aparecen de forma intermitente requieren un score promediado entre seeds
o un n_estimators mayor para estabilizarse.


## 9. Importancia de features — Permutation Importance

Shuffleamos cada feature y medimos la caída en **lift@5%** (en vez de ROC-AUC, para alinear con el objetivo real: concentrar anomalías). Caída grande = feature crítica. Caída negativa = feature posiblemente ruidosa.

In [16]:
def lift_of(feats):
    m = evaluate(feats, ALL_FEATURES, pseudo, n_estimators=SEL_N, max_features=SEL_MF)
    return m['lift@5']

base_lift = lift_of(features)
rng = np.random.RandomState(42)
perm = []
for feat in ALL_FEATURES:
    fs = features.copy()
    fs[feat] = rng.permutation(fs[feat].values)
    perm.append({'feature': feat, 'lift_drop': round(base_lift - lift_of(fs), 3)})

df_perm = pd.DataFrame(perm).sort_values('lift_drop', ascending=False)
print(f'Lift base = {base_lift:.2f}x')
print(df_perm.to_string(index=False))

Lift base = 3.70x
               feature  lift_drop
           is_external      1.852
very_high_access_ratio      1.111
     after_hours_ratio      0.741
high_plus_access_ratio      0.371
         z_exfil_peers      0.371
        z_volume_peers      0.000
       avg_session_sec      0.000
        total_accesses     -0.370
    distinct_resources     -0.370
            perm_count     -0.370
           exfil_ratio     -0.370
      z_distinct_peers     -0.370
         weekend_ratio     -0.740
     max_crit_accessed     -0.740
         z_perms_peers     -0.740
          delete_ratio     -0.740
        action_entropy     -1.111


In [17]:
d = df_perm.sort_values("lift_drop").copy()
d["signo"] = (d["lift_drop"] > 0).map({True: "baja el lift", False: "sube el lift"})
fig = px.bar(d, x="lift_drop", y="feature", orientation="h", color="signo",
             title="Permutation importance (caída de lift al shufflear cada feature)",
             color_discrete_map={"baja el lift": "#C44E52", "sube el lift": "#55A868"},
             template="plotly_white")
fig.add_vline(x=0, line_color="black")
fig.update_layout(xaxis_title="Caída en lift@5%", yaxis_title="", legend_title=""); fig.show()

## 10. Validación por consenso multi-método

El argumento más fuerte sin ground truth: si **métodos independientes** (Isolation Forest, Local Outlier Factor y un detector por suma de z-scores) coinciden en quiénes son los más anómalos, esas detecciones son robustas a la elección del algoritmo — no un artefacto de un modelo particular.

- **Isolation Forest**: aislamiento por particiones aleatorias
- **LOF**: densidad local relativa a los vecinos
- **Z-score sum**: distancia agregada a la media poblacional (Mahalanobis simplificado)

In [18]:
from sklearn.neighbors import LocalOutlierFactor

def minmax(v):
    return (v - v.min()) / (v.max() - v.min()) if v.max() > v.min() else np.zeros_like(v)

Xs = StandardScaler().fit_transform(features[ALL_FEATURES].values.astype(float))

# IF (config seleccionada)
if_s = fit_if(features, ALL_FEATURES, n_estimators=SEL_N, max_features=SEL_MF)

# LOF
lof = LocalOutlierFactor(n_neighbors=20)
lof.fit_predict(Xs)
lof_s = minmax(-lof.negative_outlier_factor_)

# Z-score sum
z_s = minmax(np.abs(Xs).sum(axis=1))

cons = pd.DataFrame({'user_id': features['user_id'], 'IF': if_s, 'LOF': lof_s, 'Zsum': z_s})
methods = ['IF', 'LOF', 'Zsum']

corr = cons[methods].corr(method='spearman')
print('Correlación de Spearman entre métodos (ranking completo):')
print(corr.round(3).to_string())

k = 25
tops = {m: set(cons.nlargest(k, m)['user_id']) for m in methods}
common = set.intersection(*tops.values())
print(f'\nUsuarios en el top-{k} de LOS TRES métodos: {len(common)} / {k}')
print(f'  {sorted(common)}')

Correlación de Spearman entre métodos (ranking completo):
         IF    LOF   Zsum
IF    1.000  0.819  0.929
LOF   0.819  1.000  0.830
Zsum  0.929  0.830  1.000

Usuarios en el top-25 de LOS TRES métodos: 13 / 25
  ['USR0009', 'USR0010', 'USR0012', 'USR0021', 'USR0030', 'USR0041', 'USR0060', 'USR0070', 'USR0096', 'USR0198', 'USR0328', 'USR0368', 'USR0420']


In [19]:
fig = px.imshow(corr, x=methods, y=methods, text_auto=".3f", color_continuous_scale="YlGn",
                zmin=0, zmax=1, title="Acuerdo entre métodos (Spearman)", template="plotly_white")
fig.show()

# Consenso = promedio de los 3 métodos; marcar quién dispara reglas
cons["consensus"] = cons[methods].mean(axis=1)
cons = cons.merge(pseudo[["user_id", "rule_score"]], on="user_id")
top_cons = cons.nlargest(15, "consensus").sort_values("consensus")
top_cons["tipo"] = (top_cons["rule_score"] > 0).map({True: "Dispara regla", False: "Solo comportamiento"})
fig = px.bar(top_cons, x="consensus", y="user_id", orientation="h", color="tipo",
             title="Top-15 por score de consenso",
             color_discrete_map={"Dispara regla": "#C44E52", "Solo comportamiento": "#4C72B0"},
             template="plotly_white")
fig.update_layout(xaxis_title="Consensus score", yaxis_title="", legend_title=""); fig.show()

n_rule = (top_cons["rule_score"] > 0).sum()
print(f"De los top-15 por consenso: {n_rule} disparan alguna regla, {15-n_rule} son solo comportamentales.")

De los top-15 por consenso: 7 disparan alguna regla, 8 son solo comportamentales.


In [20]:
# Caracterización de los anómalos de consenso que NO disparan reglas
cons_full = cons.merge(features, on='user_id').merge(
    users[['user_id', 'department', 'role', 'user_type', 'status']], on='user_id')
behav_only = cons_full.nlargest(25, 'consensus')
behav_only = behav_only[behav_only['rule_score'] == 0]

if behav_only.empty:
    print('No hay anómalos de consenso sin regla en el top-25.')
else:
    fmeans = features[ALL_FEATURES].mean()
    fstds = features[ALL_FEATURES].std(ddof=0).replace(0, 1)
    z = ((behav_only[ALL_FEATURES] - fmeans) / fstds).mean().sort_values(ascending=False)
    print(f'{len(behav_only)} anomalías de consenso SIN regla disparada.')
    print('Perfil (z-score medio vs poblacion, features mas desviadas):')
    for feat, val in z.head(6).items():
        print(f'  {feat:<26} z={val:+.2f}')
    print()
    print(behav_only[['user_id', 'department', 'role', 'user_type', 'consensus']].to_string(index=False))

15 anomalías de consenso SIN regla disparada.
Perfil (z-score medio vs poblacion, features mas desviadas):
  after_hours_ratio          z=+1.29
  delete_ratio               z=+0.71
  exfil_ratio                z=+0.50
  weekend_ratio              z=+0.43
  z_volume_peers             z=+0.38
  z_distinct_peers           z=+0.22

user_id department       role user_type  consensus
USR0030         HR    Manager  Internal   0.933537
USR0070    Fintech    Analyst  Internal   0.652245
USR0463      Legal       Reps  External   0.621802
USR0009      Legal Specialist  Internal   0.574995
USR0328         HR      Admin  Internal   0.555367
USR0198      Legal   Engineer  Internal   0.537436
USR0141      Legal Specialist  Internal   0.528572
USR0096         HR    Manager  Internal   0.505340
USR0479         HR    Analyst  Internal   0.499485
USR0420      Legal    Manager  Internal   0.494901
USR0368      Legal      Admin  Internal   0.493306
USR0155         HR    Analyst  Internal   0.479983
USR0061

## 11. Decisión final y nuevos parámetros

Comparativa baseline vs. configuración seleccionada y bloque YAML resultante.

In [21]:
comparison = pd.DataFrame([
    {'config': 'baseline', 'n_estimators': 200, 'max_features': 1.0,
     'stab_full': base_stab_full, 'stab_top20': base_stab_topk,
     'lift@5': baseline['lift@5'], 'roc_auc': baseline['roc_auc']},
    {'config': 'seleccionada', 'n_estimators': SEL_N, 'max_features': SEL_MF,
     'stab_full': SEL['stab_full'], 'stab_top20': SEL['stab_top20'],
     'lift@5': SEL['lift@5'], 'roc_auc': SEL['roc_auc']},
])
print(comparison.to_string(index=False))

      config  n_estimators  max_features  stab_full  stab_top20  lift@5  roc_auc
    baseline           200           1.0     0.9704      0.6144   4.074   0.8737
seleccionada           500           0.9     0.9884      0.7293   3.704   0.8702


### Justificación de la decisión

> Completar con valores reales tras ejecutar:
>
> - **`n_estimators` = {SEL_N}:** elegido por máxima estabilidad. ¿Hubo convergencia? ¿Vale la pena el costo de más árboles?
> - **`max_features` = {SEL_MF}:** ¿valores < 1.0 mejoraron la estabilidad sin perder lift?
> - **`contamination`:** se documenta como no-influyente en el ranking. Solo importaría si en producción se usara `predict()` para un flag binario; en ese caso conviene alinearlo a la tasa base de anomalías (~{base_rate}).
> - **Features clave** (permutation): listar top-5.
> - **Consenso:** N usuarios coinciden en el top-25 de los 3 métodos → detecciones robustas.
>
> **Limitaciones:** pseudo-labels imperfectas; dataset chico (500 usuarios); sin ground truth real de cuentas comprometidas.

## 12. Nuevos parámetros para el pipeline

`max_features` es un **parámetro nuevo** que no existía en la configuración original — hay que agregarlo tanto al YAML como al nodo `train_isolation_forest` (que actualmente no lo pasa al `IsolationForest`).

In [22]:
print('=== Bloque para conf/base/parameters/risk_scoring.yml ===\n')
print('risk_scoring:')
print('  isolation_forest:')
print(f'    n_estimators: {SEL_N}')
print(f'    max_features: {SEL_MF}        # NUEVO parametro (antes no existia)')
print(f'    max_samples: "auto"')
print(f'    contamination: 0.05         # no afecta el ranking; solo predict()')
print(f'    random_state: 42')
print()
print('Cambios vs baseline:')
print(f'  n_estimators: 200 -> {SEL_N}')
print(f'  max_features: (no existia) -> {SEL_MF}')
print()
print('IMPORTANTE: agregar max_features=if_params["max_features"] en')
print('train_isolation_forest (nodes.py), que hoy NO lo pasa al modelo.')

=== Bloque para conf/base/parameters/risk_scoring.yml ===

risk_scoring:
  isolation_forest:
    n_estimators: 500
    max_features: 0.9        # NUEVO parametro (antes no existia)
    max_samples: "auto"
    contamination: 0.05         # no afecta el ranking; solo predict()
    random_state: 42

Cambios vs baseline:
  n_estimators: 200 -> 500
  max_features: (no existia) -> 0.9

IMPORTANTE: agregar max_features=if_params["max_features"] en
train_isolation_forest (nodes.py), que hoy NO lo pasa al modelo.


## 13. Comparación final: IF-solo vs Ensemble (IF + LOF + Z-score)

Comparamos el modelo de una sola familia (Isolation Forest) contra el ensemble de tres detectores independientes, usando las métricas validadas. El ensemble es lo que finalmente se implementó en el pipeline (`score_anomaly_ensemble`).

**Hipótesis:** como LOF y Z-score son deterministas (no dependen de `random_state`), 2/3 del ensemble está anclado → la lista accionable (top-K) debería ser más reproducible que con IF-solo.

In [23]:
from sklearn.neighbors import LocalOutlierFactor

def _minmax(v):
    r = v.max() - v.min()
    return np.zeros_like(v) if r == 0 else (v - v.min()) / r

X_cmp = features[ALL_FEATURES].values.astype(float)

def score_if_only(seed):
    sc = StandardScaler().fit(X_cmp); Xs = sc.transform(X_cmp)
    m = IsolationForest(n_estimators=500, max_features=0.9, random_state=seed).fit(Xs)
    return _minmax(-m.decision_function(Xs))

def score_ensemble(seed):
    sc = StandardScaler().fit(X_cmp); Xs = sc.transform(X_cmp)
    m = IsolationForest(n_estimators=500, max_features=0.9, random_state=seed).fit(Xs)
    if_c = _minmax(-m.decision_function(Xs))
    lof = LocalOutlierFactor(n_neighbors=20); lof.fit_predict(Xs)
    lof_c = _minmax(-lof.negative_outlier_factor_)
    z_c = _minmax(np.abs(Xs).sum(axis=1))
    return (if_c + lof_c + z_c) / 3

SEEDS_CMP = [0, 7, 13, 42, 99]
base_rate = pseudo['pseudo_label'].mean()

def metric_block(score_fn):
    scores = {s: score_fn(s) for s in SEEDS_CMP}
    full, jac = [], []
    for i in range(len(SEEDS_CMP)):
        for j in range(i + 1, len(SEEDS_CMP)):
            a, b = scores[SEEDS_CMP[i]], scores[SEEDS_CMP[j]]
            full.append(spearmanr(a, b).statistic)
            ta, tb = set(np.argsort(a)[-20:]), set(np.argsort(b)[-20:])
            jac.append(len(ta & tb) / len(ta | tb))
    sc = pd.DataFrame({'user_id': features['user_id'], 'risk': scores[42]}).merge(
        pseudo[['user_id', 'pseudo_label', 'rule_score']], on='user_id')
    auc = roc_auc_score(sc['pseudo_label'], sc['risk'])
    k = int(len(sc) * 0.05)
    lift = sc.nlargest(k, 'risk')['pseudo_label'].mean() / base_rate
    rho = spearmanr(sc['rule_score'], sc['risk']).statistic
    return {'stab_full': round(np.mean(full), 4), 'stab_top20': round(np.mean(jac), 4),
            'lift@5': round(lift, 3), 'roc_auc': round(auc, 4),
            'spearman_rules': round(rho, 4)}

cmp = pd.DataFrame({'IF-solo': metric_block(score_if_only),
                    'Ensemble': metric_block(score_ensemble)}).T
cmp['delta'] = (cmp.loc['Ensemble'] - cmp.loc['IF-solo']).round(4)
print(cmp.to_string())

          stab_full  stab_top20  lift@5  roc_auc  spearman_rules  delta
IF-solo      0.9891      0.6696   3.704   0.8702          0.4009    NaN
Ensemble     0.9967      0.8788   3.704   0.8310          0.3603    NaN


### Conclusión de la comparación

| Métrica | IF-solo | Ensemble | Lectura |
|---|---|---|---|
| **Stability top-20** | ~0.67 | ~0.88 | 🟢 **+0.21** — la lista accionable es mucho más reproducible |
| Stability full | ~0.99 | ~1.00 | 🟢 marginal (ya en techo) |
| Lift@5% | 3.70x | 3.70x | ⚪ igual — no pierde concentración en el extremo |
| ROC-AUC | ~0.87 | ~0.83 | 🟠 −0.04 — concuerda menos con las reglas |
| Spearman vs reglas | ~0.40 | ~0.36 | 🟠 −0.04 — más independiente de la Capa 1 |

**Decisión: se adopta el ensemble.** Ponderando la confiabilidad de cada métrica:
- La **stability top-K es label-free** (confiable) y mejora fuerte (+0.21). Motor: LOF y Z-score son deterministas, anclan 2/3 del score.
- ROC-AUC y Spearman dependen de **pseudo-labels** (sesgadas) y bajan poco (−0.04). Menor concordancia con las reglas es deseable: la Capa 1 ya cubre lo determinista; la Capa 2 debe ser complementaria.
- El **lift intacto (3.70x)** confirma que el extremo superior sigue alineado con los casos conocidos.

## 14. Validación train/test y limitación metodológica

**Pregunta:** ¿estamos separando train de test al entrenar el modelo y calcular las métricas?

**Respuesta corta: no, y es una decisión consciente que documentamos como limitación.** El análisis abajo muestra por qué, y cuánto importa.

### ¿A qué métrica le aplica el split?

| Métrica | ¿Necesita split? | Razón |
|---|---|---|
| **Stability** (full / top-K) | ❌ No aplica | Mide reproducibilidad variando el `random_state` sobre los mismos datos. Es independiente de la generalización a datos nuevos → válida in-sample. |
| **Lift / ROC-AUC** | ⚠️ Idealmente sí | Se calculan in-sample (fit y evaluación sobre los mismos 500 usuarios) → hay un sesgo optimista leve. |

### Por qué el sesgo es menor que en un modelo supervisado

El modelo es **no supervisado**: nunca vio las pseudo-etiquetas durante el `fit` (las etiquetas vienen de las reglas, independientes del entrenamiento del IF). No hay *fuga de etiquetas* clásica — solo el efecto leve de que la frontera del IF se ajustó incluyendo esos mismos puntos.

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor

def _mm(v):
    r = v.max() - v.min()
    return np.zeros_like(v) if r == 0 else (v - v.min()) / r

dfv = features.merge(pseudo[['user_id', 'pseudo_label']], on='user_id')
base = dfv['pseudo_label'].mean()

def lift_at5(risk, label):
    k = max(1, int(len(risk) * 0.05))
    order = np.argsort(risk)[::-1][:k]
    return label[order].mean() / base

# ── IN-SAMPLE (lo que hace el pipeline): fit y score sobre todos ──
Xall = dfv[ALL_FEATURES].values.astype(float)
sc = StandardScaler().fit(Xall); Xs = sc.transform(Xall)
ifm = IsolationForest(n_estimators=500, max_features=0.9, random_state=42).fit(Xs)
lift_in = lift_at5(_mm(-ifm.decision_function(Xs)), dfv['pseudo_label'].values)

# ── HELD-OUT 70/30, 20 splits estratificados ──
lifts_oos = []
for seed in range(20):
    tr, te = train_test_split(dfv, test_size=0.3, random_state=seed,
                              stratify=dfv['pseudo_label'])
    Xtr, Xte = tr[ALL_FEATURES].values.astype(float), te[ALL_FEATURES].values.astype(float)
    s = StandardScaler().fit(Xtr)
    m = IsolationForest(n_estimators=500, max_features=0.9, random_state=42).fit(s.transform(Xtr))
    risk_te = _mm(-m.decision_function(s.transform(Xte)))
    lifts_oos.append(lift_at5(risk_te, te['pseudo_label'].values))

print(f'Tasa base de anomalías (rule-flagged): {base:.1%}  ({int(dfv["pseudo_label"].sum())} de {len(dfv)})')
print()
print(f'Lift@5% IN-SAMPLE (actual):           {lift_in:.2f}x')
print(f'Lift@5% HELD-OUT (70/30, 20 splits):  {np.mean(lifts_oos):.2f}x  +/- {np.std(lifts_oos):.2f}')
print()
n_te = int(len(dfv) * 0.3)
print(f'En un test de 30% = {n_te} usuarios, el top-5% son solo {max(1, int(n_te*0.05))} usuarios')
print('=> el held-out aleatorio es ruidoso porque parte una poblacion de anomalias ya minuscula')

Tasa base de anomalías (rule-flagged): 10.8%  (54 de 500)

Lift@5% IN-SAMPLE (actual):           3.70x
Lift@5% HELD-OUT (70/30, 20 splits):  3.11x  +/- 1.63

En un test de 30% = 150 usuarios, el top-5% son solo 7 usuarios
=> el held-out aleatorio es ruidoso porque parte una poblacion de anomalias ya minuscula


In [25]:
dfb = pd.DataFrame({
    "eval": ["In-sample (actual)", "Held-out (70/30)"],
    "lift": [lift_in, np.mean(lifts_oos)],
    "err":  [0, np.std(lifts_oos)],
})
fig = px.bar(dfb, x="eval", y="lift", error_y="err", color="eval",
             color_discrete_sequence=["#4C72B0", "#DD8452"], template="plotly_white",
             title="Optimismo in-sample: lift cae al evaluar en datos no vistos")
fig.add_hline(y=1.0, line_dash="dash", line_color="gray", annotation_text="lift=1 (azar)")
fig.update_layout(showlegend=False, xaxis_title="", yaxis_title="Lift@5%"); fig.show()

### Conclusión y limitaciones documentadas

El análisis anterior cuantifica un sesgo de evaluación, pero conviene ordenar las limitaciones por **cuánto pesan realmente**.

#### Limitación 1 (la más importante) — No existe ground truth real

No hay ni un solo usuario etiquetado como "atacante confirmado" o "cuenta comprometida". Por lo tanto **todas** las métricas de calidad (lift, ROC-AUC) se calculan contra las **reglas duras** como sustituto de la verdad.

Esto significa que lo que medimos es *"¿el modelo de ML concuerda con las reglas?"*, **no** *"¿el modelo atrapa atacantes reales?"*. Consecuencias:
- La **precisión y el recall verdaderos son imposibles de conocer** con estos datos.
- Un lift alto puede significar que el ML replica las reglas, no que descubra amenazas nuevas (por eso priorizamos métricas *label-free* como la estabilidad y la validación por consenso multi-método).
- Para resolverlo haría falta un proceso de etiquetado: incidentes confirmados por el SOC, feedback de analistas, o un dataset con ataques inyectados conocidos.

#### Limitación 2 (menor y ya cuantificada) — Evaluación in-sample

El modelo se entrena y se evalúa sobre los mismos 500 usuarios, lo que infla levemente los números absolutos. Lo medimos: el lift cae de **3.70x (in-sample) a ~3.1x (held-out)**, un optimismo de ~16%.
- No se corrigió con un `train/test split` aleatorio porque, con solo ~54 anomalías en 500 usuarios, la estimación held-out es demasiado inestable (±1.6) para ser confiable.
- Importante: este sesgo afecta a **todos los modelos por igual**, así que las conclusiones *relativas* (ensemble > IF-solo) se mantienen — solo los valores *absolutos* tienen optimismo.

#### Mejora futura (no es una limitación, es la solución correcta)

La validación rigurosa para un UEBA es un **split temporal**: entrenar la línea base con los primeros ~5 meses de `access_logs` y detectar anomalías sobre los últimos ~2 meses, simulando cómo opera en producción (puntuar comportamiento futuro contra un baseline del pasado). Requiere reconstruir las features por ventana temporal — fuera del alcance de este entregable.